In [1]:
import pandas as pd
import duckdb
import matplotlib
matplotlib.use("Agg") # Mode rendering tanpa GUI
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np
import os

# 1. KONFIGURASI PATH & WARNA

In [2]:
RAW_PATH   = "/content/superstore_powerbi.xlsx"
DB_PATH    = "/content/superstore.duckdb"
OUT_CSV    = "/content/cleaned_superstore_fact.csv"
OUT_DIR    = "/content"
EXCEL_PATH = f"{OUT_DIR}/superstore_powerbi_full.xlsx"
DASH_PATH  = f"{OUT_DIR}/dashboard.png"

# Color palette untuk Dashboard
PALETTE   = ["#2C6FBA", "#E87C3E", "#2EAC6D", "#C94040", "#7B5EA7", "#D4A017"]
BLUE      = "#2C6FBA"
ORANGE    = "#E87C3E"
GREEN     = "#2EAC6D"
RED       = "#C94040"
BG        = "#F9FAFB"
DARK      = "#1E2735"

print("✅ Konfigurasi selesai.")

✅ Konfigurasi selesai.


# 2. DATA CLEANING & FEATURE ENGINEERING

In [3]:
print("=" * 70)
print(" 🚀 STARTING END-TO-END SUPERSTORE PIPELINE")
print("=" * 70)
print("\n[1/5] Memuat dan Membersihkan Data...")

if not os.path.exists(RAW_PATH):
    raise FileNotFoundError(f"File {RAW_PATH} tidak ditemukan. Silakan unggah file terlebih dahulu.")

df = pd.read_excel(RAW_PATH)

# Standardisasi nama kolom (snake_case)
df.columns = (
    df.columns.str.strip().str.lower()
    .str.replace(r"[/ ]", "_", regex=True)
    .str.replace(r"[^a-z0-9_]", "", regex=True)
)

# Parsing tanggal
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["ship_date"]  = pd.to_datetime(df["ship_date"], errors="coerce")

# Menghapus data tanpa tanggal pesanan
df.dropna(subset=["order_date"], inplace=True)

# Ekstraksi komponen waktu & durasi pengiriman
df["order_year"]       = df["order_date"].dt.year
df["order_month"]      = df["order_date"].dt.month
df["order_quarter"]    = df["order_date"].dt.quarter
df["order_month_name"] = df["order_date"].dt.strftime("%b")
df["ship_days"]        = (df["ship_date"] - df["order_date"]).dt.days

# Validasi kolom numerik
for col in ["sales", "quantity", "discount", "profit"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Normalisasi variabel profit (sesuai format dashboard sebelumnya)
df["profit_clean"] = df["profit"]
df["profit_margin"] = (df["profit_clean"] / df["sales"]).replace([np.inf, -np.inf], 0).round(4)
df["revenue_per_qty"] = (df["sales"] / df["quantity"].replace(0, 1)).round(2)

# Klasifikasi Profit Band
def profit_segment(p):
    if pd.isna(p): return "Unknown"
    if p < 0:      return "Loss"
    elif p < 50:   return "Low Profit"
    elif p < 200:  return "Mid Profit"
    else:          return "High Profit"
df["profit_band"] = df["profit_clean"].apply(profit_segment)

# Klasifikasi Discount Bucket
def categorize_discount(d):
    if pd.isna(d) or d == 0: return 'No Discount'
    elif d <= 0.10: return '1-10%'
    elif d <= 0.20: return '11-20%'
    elif d <= 0.30: return '21-30%'
    else: return 'Above 30%'
df["discount_bucket"] = df["discount"].apply(categorize_discount)

# Menyimpan data bersih (Optional backup)
df.to_csv(OUT_CSV, index=False)
print(f"      ✓ Data bersih selesai ({len(df):,} baris). Tersimpan di {OUT_CSV}")

 🚀 STARTING END-TO-END SUPERSTORE PIPELINE

[1/5] Memuat dan Membersihkan Data...
      ✓ Data bersih selesai (10,194 baris). Tersimpan di /content/cleaned_superstore_fact.csv


# 3. DUCKDB SQL AGGREGATIONS

In [4]:
print("\n[2/5] Memproses Analisis SQL via DuckDB...")

con = duckdb.connect(DB_PATH)
con.execute("CREATE OR REPLACE TABLE orders AS SELECT * FROM df")

# Agregasi 1: Sales by Year & Region
by_yr_reg = con.execute("""
    SELECT order_year, region,
           COUNT(DISTINCT order_id) AS total_orders, COUNT(*) AS total_items,
           SUM(sales) AS total_sales, SUM(profit_clean) AS total_profit, AVG(discount) AS avg_discount
    FROM orders GROUP BY order_year, region
""").df()
by_yr_reg["profit_margin_pct"] = (by_yr_reg["total_profit"] / by_yr_reg["total_sales"] * 100).round(1)

# Agregasi 2: Sales by Category / Sub-Category
by_cat = con.execute("""
    SELECT category, subcategory,
           SUM(sales) AS total_sales, SUM(profit_clean) AS total_profit,
           SUM(quantity) AS total_qty, COUNT(DISTINCT order_id) AS total_orders
    FROM orders GROUP BY category, subcategory
""").df()
by_cat["profit_margin_pct"] = (by_cat["total_profit"] / by_cat["total_sales"] * 100).round(1)

# Agregasi 3: Monthly Trend
by_month = con.execute("""
    SELECT order_year, order_quarter, order_month, order_month_name,
           SUM(sales) AS monthly_sales, SUM(profit_clean) AS monthly_profit, COUNT(DISTINCT order_id) AS order_count
    FROM orders GROUP BY order_year, order_quarter, order_month, order_month_name
    ORDER BY order_year, order_month
""").df()

# Agregasi 4: Customer Segment
by_seg = con.execute("""
    SELECT segment, COUNT(DISTINCT customer_id) AS unique_customers, COUNT(DISTINCT order_id) AS total_orders,
           SUM(sales) AS total_sales, SUM(profit_clean) AS total_profit
    FROM orders GROUP BY segment
""").df()
by_seg["profit_margin_pct"] = (by_seg["total_profit"] / by_seg["total_sales"] * 100).round(1)
by_seg["avg_sales_per_customer"] = (by_seg["total_sales"] / by_seg["unique_customers"]).round(0)

# Agregasi 5: Discount Impact
by_disc = con.execute("""
    SELECT discount_bucket, COUNT(order_id) AS order_count, AVG(sales) AS avg_sales, AVG(profit_clean) AS avg_profit,
           SUM(profit_clean) AS total_profit, SUM(sales) AS total_sales
    FROM orders GROUP BY discount_bucket
""").df()
by_disc["profit_margin_pct"] = (by_disc["total_profit"] / by_disc["total_sales"] * 100).round(1)

# Agregasi 6: Shipping
by_ship = con.execute("""
    SELECT ship_mode, AVG(ship_days) AS avg_ship_days, COUNT(DISTINCT order_id) AS total_orders,
           SUM(sales) AS total_sales, SUM(profit_clean) AS total_profit
    FROM orders GROUP BY ship_mode ORDER BY avg_ship_days
""").df()
by_ship["profit_margin_pct"] = (by_ship["total_profit"] / by_ship["total_sales"] * 100).round(1)

# Agregasi 7: Top States
by_state = con.execute("""
    SELECT state_province, region, SUM(sales) AS total_sales, SUM(profit_clean) AS total_profit,
           COUNT(DISTINCT order_id) AS total_orders
    FROM orders GROUP BY state_province, region ORDER BY total_sales DESC
""").df()
by_state["profit_margin_pct"] = (by_state["total_profit"] / by_state["total_sales"] * 100).round(1)

# Agregasi 8: Loss-Making Orders (RCA)
by_loss_making = con.execute("""
    SELECT order_id, customer_name, segment, state_province, category, subcategory, product_name,
           ROUND(sales, 2) AS sales, ROUND(profit_clean, 2) AS profit, ROUND(discount * 100, 0) AS discount_pct, profit_band
    FROM orders WHERE profit_clean < 0 ORDER BY profit_clean ASC LIMIT 500
""").df()

# Agregasi 9: Customer RFM Value
by_rfm = con.execute("""
    SELECT customer_id, customer_name, segment, COUNT(DISTINCT order_id) AS frequency,
           ROUND(SUM(sales), 2) AS monetary, ROUND(SUM(profit_clean), 2) AS total_profit,
           MAX(order_date) AS last_order_date, ROUND(SUM(sales) / COUNT(DISTINCT order_id), 0) AS avg_order_value
    FROM orders GROUP BY customer_id, customer_name, segment ORDER BY monetary DESC
""").df()
by_rfm["last_order_date"] = by_rfm["last_order_date"].astype(str) # Fix untuk format Excel

con.close()
print("      ✓ Ekstraksi agregasi SQL selesai.")


[2/5] Memproses Analisis SQL via DuckDB...
      ✓ Ekstraksi agregasi SQL selesai.


# 4. GENERATE DASHBOARD (MATPLOTLIB)

In [5]:
print("\n[3/5] Membuat Visualisasi Dashboard PNG...")

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.facecolor": BG,
    "figure.facecolor": DARK, "axes.edgecolor": "#404B5A", "axes.labelcolor": "#CDD6E0",
    "xtick.color": "#CDD6E0", "ytick.color": "#CDD6E0", "text.color": "white",
    "grid.color": "#2E3848", "grid.alpha": 0.6, "axes.grid": True, "axes.grid.axis": "y"
})

fig = plt.figure(figsize=(22, 16), facecolor=DARK)
fig.suptitle("Sample Superstore — Executive Dashboard", fontsize=20, fontweight="bold", color="white", y=0.98)
gs = gridspec.GridSpec(4, 4, figure=fig, hspace=0.55, wspace=0.4, top=0.93, bottom=0.04, left=0.05, right=0.97)

def fmt_m(v): return f"${v/1e6:.1f}M"
def ax_style(ax, title):
    ax.set_facecolor(BG)
    ax.set_title(title, fontsize=10, fontweight="bold", color="white", pad=8)
    for spine in ax.spines.values(): spine.set_edgecolor("#404B5A")

# [KPI Cards]
kpi_ax = [fig.add_subplot(gs[0, c]) for c in range(4)]
kpis = [
    ("💰 Total Revenue", fmt_m(df["sales"].sum()), BLUE),
    ("📈 Total Profit",  fmt_m(df["profit_clean"].sum()), GREEN),
    ("🧾 Total Orders",  f"{df['order_id'].nunique():,}", ORANGE),
    ("📦 Avg Ship Days", f"{df['ship_days'].mean():.1f} days", "#7B5EA7"),
]
for ax, (title, val, color) in zip(kpi_ax, kpis):
    ax.set_facecolor(DARK)
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.set_xticks([]); ax.set_yticks([])
    ax.add_patch(mpatches.FancyBboxPatch((0.05, 0.1), 0.9, 0.8, boxstyle="round,pad=0.02", facecolor=color+"22", edgecolor=color, linewidth=1.5, transform=ax.transAxes))
    ax.text(0.5, 0.65, val, ha="center", va="center", transform=ax.transAxes, fontsize=18, fontweight="bold", color=color)
    ax.text(0.5, 0.25, title, ha="center", va="center", transform=ax.transAxes, fontsize=9, color="#CDD6E0")

# [Chart 1: Monthly Trend]
ax1 = fig.add_subplot(gs[1, :2])
ax_style(ax1, "📅 Monthly Sales & Profit Trend")
trend = by_month[by_month["order_year"] >= 2024].copy()
trend["period"] = trend["order_year"].astype(str) + "-" + trend["order_month"].astype(str).str.zfill(2)
x = range(len(trend))
ax1.fill_between(x, trend["monthly_sales"]/1e3, alpha=0.3, color=BLUE)
ax1.plot(x, trend["monthly_sales"]/1e3, color=BLUE, linewidth=2, label="Sales ($K)")
ax2b = ax1.twinx()
ax2b.plot(x, trend["monthly_profit"]/1e3, color=GREEN, linewidth=2, linestyle="--", label="Profit ($K)")
ax2b.tick_params(colors="#CDD6E0")
ax2b.set_ylabel("Profit ($K)", color=GREEN, fontsize=8)
ax2b.spines["right"].set_edgecolor("#404B5A")
ax2b.set_facecolor(BG)
step = max(1, len(trend)//8)
ax1.set_xticks(list(range(0, len(trend), step)))
ax1.set_xticklabels([trend["period"].iloc[i] for i in range(0, len(trend), step)], rotation=45, fontsize=7)
ax1.set_ylabel("Sales ($K)", color=BLUE, fontsize=8)
lines1, labels1 = ax1.get_legend_handles_labels(); lines2, labels2 = ax2b.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc="upper left", fontsize=8, facecolor="#1E2735", edgecolor="#404B5A", labelcolor="white")

# [Chart 2: Sales by Category]
ax2 = fig.add_subplot(gs[1, 2])
ax_style(ax2, "🛒 Sales by Category")
cat_grp = df.groupby("category")["sales"].sum().sort_values(ascending=True)
bars = ax2.barh(cat_grp.index, cat_grp.values/1e6, color=[ORANGE, BLUE, GREEN], height=0.5)
for bar, val in zip(bars, cat_grp.values): ax2.text(val/1e6 + 0.01, bar.get_y() + bar.get_height()/2, fmt_m(val), va="center", fontsize=8, color="white")
ax2.set_xlabel("Sales ($M)", fontsize=8); ax2.grid(axis="x"); ax2.grid(axis="y", visible=False)

# [Chart 3: Segment Donut]
ax3 = fig.add_subplot(gs[1, 3])
ax_style(ax3, "👥 Revenue by Segment")
ax3.set_facecolor(DARK)
for spine in ax3.spines.values(): spine.set_visible(False)
ax3.set_xticks([]); ax3.set_yticks([])
seg_sales = df.groupby("segment")["sales"].sum()
wedges, texts, autotexts = ax3.pie(seg_sales, labels=seg_sales.index, autopct="%1.1f%%", colors=[BLUE, ORANGE, GREEN], startangle=90, wedgeprops={"width": 0.5, "edgecolor": DARK, "linewidth": 2}, pctdistance=0.75)
for t in texts: t.set_fontsize(8); t.set_color("white")
for t in autotexts: t.set_fontsize(8); t.set_color("white")
ax3.set_facecolor(DARK)

# [Chart 4: Subcategory Profit]
ax4 = fig.add_subplot(gs[2, :2])
ax_style(ax4, "📊 Profit by Sub-Category")
sub_profit = df.groupby("subcategory")["profit_clean"].sum().sort_values()
ax4.barh(sub_profit.index, sub_profit.values/1e3, color=[RED if v < 0 else GREEN for v in sub_profit.values], height=0.6)
ax4.axvline(0, color="white", linewidth=0.8, alpha=0.5)
ax4.set_xlabel("Profit ($K)", fontsize=8); ax4.grid(axis="x"); ax4.grid(axis="y", visible=False)

# [Chart 5: Discount Impact]
ax5 = fig.add_subplot(gs[2, 2])
ax_style(ax5, "🏷️ Discount Impact on Profit")
disc = by_disc.copy()
bars5 = ax5.bar(disc["discount_bucket"].astype(str), disc["profit_margin_pct"], color=[GREEN if v >= 0 else RED for v in disc["profit_margin_pct"]], width=0.6)
ax5.axhline(0, color="white", linewidth=0.8, alpha=0.5)
ax5.set_ylabel("Profit Margin %", fontsize=8); ax5.set_xlabel("Discount Range", fontsize=8)
plt.setp(ax5.get_xticklabels(), rotation=30, ha="right", fontsize=7)
for bar in bars5: ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (1 if bar.get_height() >= 0 else -3), f"{bar.get_height():.0f}%", ha="center", va="bottom", fontsize=7, color="white")

# [Chart 6: Region]
ax6 = fig.add_subplot(gs[2, 3])
ax_style(ax6, "🗺️ Sales & Profit by Region")
reg = df.groupby("region").agg(total_sales=("sales","sum"), total_profit=("profit_clean","sum")).reset_index().sort_values("total_sales", ascending=False)
x_pos = np.arange(len(reg)); width = 0.35
ax6.bar(x_pos - width/2, reg["total_sales"]/1e6, width, label="Sales", color=BLUE, alpha=0.85)
ax6_r = ax6.twinx()
ax6_r.bar(x_pos + width/2, reg["total_profit"]/1e6, width, label="Profit", color=GREEN, alpha=0.85)
ax6_r.tick_params(colors="#CDD6E0"); ax6_r.set_ylabel("Profit ($M)", color=GREEN, fontsize=8)
ax6_r.spines["right"].set_edgecolor("#404B5A"); ax6_r.set_facecolor(BG)
ax6.set_xticks(x_pos); ax6.set_xticklabels(reg["region"], fontsize=8); ax6.set_ylabel("Sales ($M)", color=BLUE, fontsize=8); ax6.grid(axis="y")

# [Chart 7: Ship Mode]
ax7 = fig.add_subplot(gs[3, :2])
ax_style(ax7, "🚚 Shipping Mode Performance")
ship_yr = df.groupby(["order_year","ship_mode"])["sales"].sum().reset_index()
years_avail = sorted(ship_yr["order_year"].unique())[-3:]
ship_pivot = ship_yr[ship_yr["order_year"].isin(years_avail)].pivot(index="ship_mode", columns="order_year", values="sales").fillna(0)
x_s = np.arange(len(ship_pivot)); n_years = len(ship_pivot.columns); w = 0.7 / n_years
for i, (yr, col) in enumerate(zip(ship_pivot.columns, PALETTE)): ax7.bar(x_s + (i - n_years/2 + 0.5)*w, ship_pivot[yr]/1e6, w, label=str(yr), color=col, alpha=0.85)
ax7.set_xticks(x_s); ax7.set_xticklabels(ship_pivot.index, fontsize=8); ax7.set_ylabel("Sales ($M)", fontsize=8)
ax7.legend(fontsize=8, facecolor="#1E2735", edgecolor="#404B5A", labelcolor="white"); ax7.grid(axis="y")

# [Chart 8: YoY Growth]
ax8 = fig.add_subplot(gs[3, 2:])
ax_style(ax8, "📈 Year-over-Year Sales Growth")
yr_total = df.groupby("order_year")["sales"].sum().reset_index()
yr_total["yoy_growth"] = yr_total["sales"].pct_change() * 100
ax8.bar(yr_total["order_year"].astype(str), yr_total["sales"]/1e6, color=BLUE, alpha=0.6, label="Revenue")
ax8_r = ax8.twinx()
yoy_clean = yr_total.dropna(subset=["yoy_growth"])
ax8_r.plot(yoy_clean["order_year"].astype(str), yoy_clean["yoy_growth"], color=ORANGE, marker="o", linewidth=2, markersize=6, label="YoY %")
ax8_r.axhline(0, color="#404B5A", linewidth=0.8)
ax8_r.tick_params(colors="#CDD6E0"); ax8_r.set_ylabel("YoY Growth %", color=ORANGE, fontsize=8)
ax8_r.spines["right"].set_edgecolor("#404B5A"); ax8_r.set_facecolor(BG)
ax8.set_ylabel("Revenue ($M)", color=BLUE, fontsize=8); ax8.grid(axis="y")
lines_a, lbl_a = ax8.get_legend_handles_labels(); lines_b, lbl_b = ax8_r.get_legend_handles_labels()
ax8.legend(lines_a+lines_b, lbl_a+lbl_b, fontsize=8, facecolor="#1E2735", edgecolor="#404B5A", labelcolor="white")

fig.text(0.5, 0.01, "Data: Sample Superstore | Pipeline: Pandas + DuckDB", ha="center", fontsize=8, color="#7B8FA0")
fig.savefig(DASH_PATH, dpi=150, bbox_inches="tight", facecolor=DARK)
plt.close()
print(f"      ✓ Dashboard tersimpan di {DASH_PATH}")


[3/5] Membuat Visualisasi Dashboard PNG...


/tmp/ipykernel_12984/776125755.py:135: UserWarning: Glyph 128176 (\N{MONEY BAG}) missing from font(s) DejaVu Sans.
  fig.savefig(DASH_PATH, dpi=150, bbox_inches="tight", facecolor=DARK)
/tmp/ipykernel_12984/776125755.py:135: UserWarning: Glyph 128200 (\N{CHART WITH UPWARDS TREND}) missing from font(s) DejaVu Sans.
  fig.savefig(DASH_PATH, dpi=150, bbox_inches="tight", facecolor=DARK)
/tmp/ipykernel_12984/776125755.py:135: UserWarning: Glyph 129534 (\N{RECEIPT}) missing from font(s) DejaVu Sans.
  fig.savefig(DASH_PATH, dpi=150, bbox_inches="tight", facecolor=DARK)
/tmp/ipykernel_12984/776125755.py:135: UserWarning: Glyph 128230 (\N{PACKAGE}) missing from font(s) DejaVu Sans.
  fig.savefig(DASH_PATH, dpi=150, bbox_inches="tight", facecolor=DARK)
/tmp/ipykernel_12984/776125755.py:135: UserWarning: Glyph 128197 (\N{CALENDAR}) missing from font(s) DejaVu Sans.
  fig.savefig(DASH_PATH, dpi=150, bbox_inches="tight", facecolor=DARK)
/tmp/ipykernel_12984/776125755.py:135: UserWarning: Glyph 12

      ✓ Dashboard tersimpan di /content/dashboard.png


# 5. EXPORT EXCEL UNTUK POWER BI

In [6]:
print("\n[4/5] Mengekspor Data ke Excel (Power BI Format)...")

with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
    df_export = df.copy()
    df_export["order_date"] = df_export["order_date"].astype(str)
    df_export["ship_date"]  = df_export["ship_date"].astype(str)
    df_export.to_excel(writer, sheet_name="Orders_Fact", index=False)

    # Aggregation Sheets
    by_yr_reg.to_excel(writer, sheet_name="Sales_Year_Region", index=False)
    by_cat.to_excel(writer, sheet_name="Category_Performance", index=False)
    by_month.to_excel(writer, sheet_name="Monthly_Trend", index=False)
    by_seg.to_excel(writer, sheet_name="Segment_Analysis", index=False)
    by_disc.to_excel(writer, sheet_name="Discount_Impact", index=False)
    by_ship.to_excel(writer, sheet_name="Shipping_Analysis", index=False)
    by_state.to_excel(writer, sheet_name="State_Performance", index=False)
    by_loss_making.to_excel(writer, sheet_name="Loss_Making_Orders", index=False)
    by_rfm.to_excel(writer, sheet_name="Customer_RFM_Value", index=False)

print(f"      ✓ Excel file tersimpan di {EXCEL_PATH}")


[4/5] Mengekspor Data ke Excel (Power BI Format)...
      ✓ Excel file tersimpan di /content/superstore_powerbi_full.xlsx


# 6. BUSINESS INSIGHTS SUMMARY

In [7]:
print("\n[5/5] Ringkasan Eksekutif Bisnis")
print("=" * 70)

total_sales  = df["sales"].sum()
total_profit = df["profit_clean"].sum()
total_orders = df["order_id"].nunique()

print(f"📊 OVERALL PERFORMANCE")
print(f"   Revenue     : ${total_sales/1e6:.2f}M")
print(f"   Profit      : ${total_profit/1e6:.2f}M ({total_profit/total_sales*100:.1f}% margin)")
print(f"   Orders      : {total_orders:,}")

print(f"\n📦 CATEGORY BREAKDOWN")
cat_s = df.groupby("category").agg(s=("sales","sum"), p=("profit_clean","sum"))
for cat, row in cat_s.iterrows():
    print(f"   - {cat:<20} Sales: {fmt_m(row.s):<10} Margin: {row.p/row.s*100:.1f}%")

print(f"\n🏆 TOP 3 STATES BY REVENUE")
top3 = by_state.head(3)
for _, r in top3.iterrows():
    print(f"   - {r['state_province']:<20} ${r['total_sales']/1e6:.2f}M  (Margin {r['profit_margin_pct']:.1f}%)")

print(f"\n🏷️ DISCOUNT IMPACT SUMMARY")
for _, r in by_disc.iterrows():
    icon = "✅" if r["profit_margin_pct"] >= 0 else "❌"
    print(f"   {icon} {str(r['discount_bucket']):<15} Margin: {r['profit_margin_pct']:.1f}%")

print("\n" + "=" * 70)
print(" ✅ PIPELINE COMPLETE!")
print(f" 📂 Data Bersih (CSV): {OUT_CSV}")
print(f" 🗄️ Database DuckDB  : {DB_PATH}")
print(f" 📈 Dashboard Image  : {DASH_PATH}")
print(f" 📊 Excel Power BI   : {EXCEL_PATH}")
print("=" * 70)


[5/5] Ringkasan Eksekutif Bisnis
📊 OVERALL PERFORMANCE
   Revenue     : $1011.45M
   Profit      : $310.17M (30.7% margin)
   Orders      : 5,111

📦 CATEGORY BREAKDOWN
   - Furniture            Sales: $521.7M    Margin: -2.3%
   - Office Supplies      Sales: $212.4M    Margin: 64.5%
   - Technology           Sales: $277.4M    Margin: 66.7%

🏆 TOP 3 STATES BY REVENUE
   - Texas                $228.83M  (Margin -19.2%)
   - California           $225.81M  (Margin 73.4%)
   - New York             $109.08M  (Margin 100.1%)

🏷️ DISCOUNT IMPACT SUMMARY
   ✅ 11-20%          Margin: 75.2%
   ✅ 1-10%           Margin: 112.0%
   ❌ Above 30%       Margin: -42.9%
   ✅ No Discount     Margin: 127.7%
   ❌ 21-30%          Margin: -65.6%

 ✅ PIPELINE COMPLETE!
 📂 Data Bersih (CSV): /content/cleaned_superstore_fact.csv
 🗄️ Database DuckDB  : /content/superstore.duckdb
 📈 Dashboard Image  : /content/dashboard.png
 📊 Excel Power BI   : /content/superstore_powerbi_full.xlsx
